In [6]:
#!/usr/bin/env python3
"""
Metal-site pipeline (functions only, no classes)

 1. Get PDB structures that contain metal M with resolution <= MAX_RESOLUTION
    (either download them from RCSB, or scan a folder of .pdb files you have)
 2. Cut every M coordination site out into a .lig file:  pdbid_M_CN.lig
 3. CN 4  -> sort into tetrahedral (Td) and square planar (SP)
    CN 6  -> keep octahedral (Oh)
 4. For each CN / geometry, count which residues and atoms are the ligands

Only numpy is needed.  Edit the SETTINGS block and run:
    python metal_sites.py
"""
import csv
import itertools
import json
import os
import re
import shutil
import urllib.error
import urllib.request
from collections import Counter

import numpy as np

# =============================== SETTINGS ===================================
METAL = "ZN"               # element symbol of the metal, e.g. "MN", "FE", "CU", "ZN"
MAX_RESOLUTION = 1.8       # Å
DOWNLOAD = True            # True: search + download from RCSB; False: use PDB_DIR only
MAX_DOWNLOADS = 200        # limit on how many structures to download (None = all)
PDB_DIR = "pdb_files"      # where .pdb files are (or will be downloaded to)
OUT_DIR = "metal_sites_out"
CUTOFF = 2.8               # Å, max metal–donor distance counted as coordinating
DONORS = {"N", "O", "S", "SE"}
ONLY_PROTEIN_AND_WATER = True  # skip sites with any non-amino-acid / non-water ligand
MAX_CSHM = 3.0             # sites worse than this are not assigned Td / SP / Oh

# RCSB chemical component codes for each metal's ions (used for the search)
ION_CODES = {
    "MN": ["MN", "MN3"], "FE": ["FE", "FE2"], "CO": ["CO", "3CO"], "NI": ["NI", "3NI"],
    "CU": ["CU", "CU1"], "ZN": ["ZN"], "V": ["V"], "CR": ["CR"], "TI": ["TI"],
}
WATER = {"HOH", "WAT", "DOD"}
AMINO_ACIDS = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
    "MSE", "SEC", "CSO", "CSD", "KCX", "SEP", "TPO", "PTR", "OCS", "MLY",
}
# ============================================================================


# ---------------------------------------------------------------- step 1
def search_rcsb(metal, max_res):
    """Return PDB IDs (X-ray) containing the metal ion with resolution <= max_res."""
    codes = ION_CODES.get(metal.upper(), [metal.upper()])
    query = {
        "query": {"type": "group", "logical_operator": "and", "nodes": [
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_entry_info.resolution_combined",
                "operator": "less_or_equal", "value": max_res}},
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_nonpolymer_entity_container_identifiers.nonpolymer_comp_id",
                "operator": "in", "value": codes}},
        ]},
        "return_type": "entry",
        "request_options": {"return_all_hits": True},
    }
    req = urllib.request.Request("https://search.rcsb.org/rcsbsearch/v2/query",
                                 data=json.dumps(query).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            if resp.status == 204:          # query OK but nothing found
                return []
            data = json.load(resp)
    except urllib.error.HTTPError as e:
        print("RCSB search failed:", e.code, e.read().decode(errors="replace")[:500])
        return []
    return [hit["identifier"] for hit in data.get("result_set", [])]


def download_pdb(pdb_id, folder):
    """Download one structure: .pdb if it exists, otherwise .cif (mmCIF).
    Many newer / large entries are only distributed as mmCIF."""
    for ext in ("pdb", "cif"):
        path = os.path.join(folder, f"{pdb_id.lower()}.{ext}")
        if os.path.exists(path):
            return path
    for ext in ("pdb", "cif"):
        path = os.path.join(folder, f"{pdb_id.lower()}.{ext}")
        url = f"https://files.rcsb.org/download/{pdb_id.upper()}.{ext}"
        try:
            urllib.request.urlretrieve(url, path)
            return path
        except urllib.error.HTTPError as e:
            if e.code != 404:
                print(f"  could not download {pdb_id}.{ext}: {e}")
        except Exception as e:
            print(f"  could not download {pdb_id}.{ext}: {e}")
    print(f"  {pdb_id}: not available as .pdb or .cif")
    return None


# ---- mmCIF reading (simple: only what we need) ----
def cif_tokens(line):
    """Split an mmCIF data line into tokens, respecting 'quoted' and "quoted" values."""
    return [t[1:-1] if t[:1] in "'\"" else t
            for t in re.findall(r"'[^']*'(?=\s|$)|\"[^\"]*\"(?=\s|$)|\S+", line)]


def cif_value(path, keys):
    """Return the first numeric value found for any of the given single-value keys."""
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if len(parts) >= 2 and parts[0] in keys:
                try:
                    return float(parts[1])
                except ValueError:
                    pass
    return None


def pdb_line(record, serial, name, alt, resname, chain, resseq, icode, xyz, occ, b, elem):
    """Write one atom as a PDB-format line (used for atoms read from mmCIF)."""
    name = name if len(name) == 4 or len(elem) == 2 else " " + name
    return (f"{record:<6}{serial % 100000:5d} {name:<4}{alt or ' ':1}{resname:>3} "
            f"{chain[:1]:1}{resseq:>4}{icode or ' ':1}   {xyz[0]:8.3f}{xyz[1]:8.3f}{xyz[2]:8.3f}"
            f"{occ:6.2f}{b:6.2f}          {elem:>2}")


def read_atoms_cif(path):
    """Read _atom_site from an mmCIF file (first model, first alt conformer)."""
    atoms, alt_seen, cols, in_loop, model1 = [], {}, [], False, None
    with open(path) as fh:
        for line in fh:
            s = line.strip()
            if s.startswith("_atom_site."):
                cols.append(s.split()[0][len("_atom_site."):])
                in_loop = True
                continue
            if not in_loop or not cols:
                continue
            if not s or s.startswith(("#", "loop_", "_")):
                if atoms or s.startswith(("#", "loop_")):
                    break
                continue
            v = dict(zip(cols, cif_tokens(s)))
            if v.get("group_PDB") not in ("ATOM", "HETATM"):
                continue
            model = v.get("pdbx_PDB_model_num", "1")
            model1 = model1 or model
            if model != model1:
                break
            clean = lambda x: "" if x in (".", "?") else x
            name = v.get("auth_atom_id", v.get("label_atom_id"))
            resname = v.get("auth_comp_id", v.get("label_comp_id"))
            chain = v.get("auth_asym_id", v.get("label_asym_id"))
            resseq = v.get("auth_seq_id", v.get("label_seq_id"))
            icode = clean(v.get("pdbx_PDB_ins_code", ""))
            alt = clean(v.get("label_alt_id", ""))
            key = (chain, resseq + icode, resname, name)
            if alt and alt_seen.setdefault(key, alt) != alt:
                continue
            elem = v.get("type_symbol", name[:1]).upper()
            xyz = np.array([float(v["Cartn_x"]), float(v["Cartn_y"]), float(v["Cartn_z"])])
            occ = float(clean(v.get("occupancy", "1")) or 1)
            b = float(clean(v.get("B_iso_or_equiv", "0")) or 0)
            atoms.append({
                "line": pdb_line(v["group_PDB"], int(v.get("id", 0)), name, alt, resname,
                                 chain, resseq, icode, xyz, occ, b, elem),
                "record": v["group_PDB"], "name": name, "resname": resname,
                "chain": chain, "resseq": resseq + icode, "element": elem, "xyz": xyz,
            })
    return atoms


def get_resolution(pdb_path):
    """Resolution in Å; None if absent (e.g. NMR). Works for .pdb and .cif."""
    if pdb_path.lower().endswith(".cif"):
        return cif_value(pdb_path, ("_refine.ls_d_res_high", "_reflns.d_resolution_high",
                                    "_em_3d_reconstruction.resolution"))
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith("REMARK   2 RESOLUTION."):
                try:
                    return float(line[23:30])
                except ValueError:
                    return None
            if line.startswith("ATOM"):
                break
    return None


def read_atoms(pdb_path):
    """Return a list of atom dicts from the first model, first alt conformer only."""
    if pdb_path.lower().endswith(".cif"):
        return read_atoms_cif(pdb_path)
    atoms, alt_seen = [], {}
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith("ENDMDL"):
                break
            if not line.startswith(("ATOM", "HETATM")):
                continue
            alt = line[16].strip()
            name = line[12:16].strip()
            key = (line[21], line[22:27], line[17:20], name)
            if alt and alt_seen.setdefault(key, alt) != alt:
                continue
            elem = line[76:78].strip().upper()
            if not elem:
                elem = "".join(c for c in name if c.isalpha())[:2].upper()
            atoms.append({
                "line": line.rstrip("\n"), "record": line[:6].strip(), "name": name,
                "resname": line[17:20].strip(), "chain": line[21].strip(),
                "resseq": line[22:27].strip(), "element": elem,
                "xyz": np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])]),
            })
    return atoms


def contains_metal(atoms, metal):
    return any(a["record"] == "HETATM" and a["element"] == metal for a in atoms)


def collect_structures():
    """Step 1: return list of .pdb paths with the metal and resolution <= cutoff."""
    os.makedirs(PDB_DIR, exist_ok=True)
    if DOWNLOAD:
        ids = search_rcsb(METAL, MAX_RESOLUTION)
        print(f"RCSB: {len(ids)} entries with {METAL} at <= {MAX_RESOLUTION} Å")
        if MAX_DOWNLOADS:
            ids = ids[:MAX_DOWNLOADS]
        for i, pid in enumerate(ids, 1):
            print(f"  downloading {pid} ({i}/{len(ids)})", end="\r")
            download_pdb(pid, PDB_DIR)
        print()

    kept = []
    for fn in sorted(os.listdir(PDB_DIR)):
        if not fn.lower().endswith((".pdb", ".ent", ".cif")):
            continue
        path = os.path.join(PDB_DIR, fn)
        res = get_resolution(path)
        if res is None or res > MAX_RESOLUTION:
            continue
        if contains_metal(read_atoms(path), METAL):
            kept.append(path)
    print(f"Step 1: {len(kept)} structures contain {METAL} with resolution <= {MAX_RESOLUTION} Å")
    return kept


# ---------------------------------------------------------------- step 2
def find_sites(atoms, metal, cutoff):
    """Return list of (metal_atom, [ligand atoms sorted by distance])."""
    xyz = np.array([a["xyz"] for a in atoms])
    sites = []
    for i, m in enumerate(atoms):
        if m["record"] != "HETATM" or m["element"] != metal:
            continue
        d = np.linalg.norm(xyz - m["xyz"], axis=1)
        ligs = []
        for j in np.where(d <= cutoff)[0]:
            if j != i and atoms[j]["element"] in DONORS:
                ligs.append(dict(atoms[j], dist=float(d[j])))
        ligs.sort(key=lambda a: a["dist"])
        sites.append((m, ligs))
    return sites


def ligands_allowed(ligs):
    for a in ligs:
        if a["resname"] not in AMINO_ACIDS and a["resname"] not in WATER:
            return False
    return True


def write_lig(metal_atom, ligs, path):
    with open(path, "w") as fh:
        fh.write(metal_atom["line"] + "\n")
        for a in ligs:
            fh.write(a["line"] + "\n")


def unique_name(folder, base):
    """pdbid_M_CN.lig, or pdbid_M_CN_2.lig, _3 ... if a site with that name exists."""
    name, k = f"{base}.lig", 2
    while os.path.exists(os.path.join(folder, name)):
        name, k = f"{base}_{k}.lig", k + 1
    return name


def extract_sites(pdb_paths):
    """Step 2: write .lig files; return a list of site dicts."""
    lig_dir = os.path.join(OUT_DIR, "lig_files")
    os.makedirs(lig_dir, exist_ok=True)
    sites, skipped = [], 0
    for path in pdb_paths:
        pdb_id = os.path.basename(path).split(".")[0].lower()
        for metal_atom, ligs in find_sites(read_atoms(path), METAL, CUTOFF):
            if not ligs or (ONLY_PROTEIN_AND_WATER and not ligands_allowed(ligs)):
                skipped += 1
                continue
            cn = len(ligs)
            fname = unique_name(lig_dir, f"{pdb_id}_{METAL}_{cn}")
            write_lig(metal_atom, ligs, os.path.join(lig_dir, fname))
            sites.append({"file": fname, "pdb": pdb_id, "cn": cn,
                          "metal": metal_atom, "ligands": ligs})
    print(f"Step 2: {len(sites)} sites written to {lig_dir}  ({skipped} skipped)")
    return sites


# ---------------------------------------------------------------- step 3
def unit(v):
    v = np.asarray(v, float)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


IDEAL = {
    "Td": unit([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]),
    "SP": unit([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0]]),
    "Oh": unit([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]]),
}


def cshm(vectors, ideal):
    """Continuous shape measure (metal included). 0 = perfect shape."""
    Q = np.vstack([np.zeros(3), vectors])
    Q = Q - Q.mean(0)
    best = np.inf
    for perm in itertools.permutations(range(len(vectors))):
        P = np.vstack([np.zeros(3), ideal[list(perm)]])
        P = P - P.mean(0)
        U, S, Vt = np.linalg.svd(P.T @ Q)
        d = np.sign(np.linalg.det(Vt.T @ U.T))
        tr = S[0] + S[1] + d * S[2]
        best = min(best, 100 * (1 - tr ** 2 / ((Q ** 2).sum() * (P ** 2).sum())))
    return max(best, 0.0)


def bond_angles(vectors):
    out = []
    for u, v in itertools.combinations(vectors, 2):
        c = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
        out.append(float(np.degrees(np.arccos(np.clip(c, -1, 1)))))
    return out


def tau4(angles):
    a = sorted(angles, reverse=True)
    return (360 - (a[0] + a[1])) / 141          # 1 = Td, 0 = SP


def classify_site(site):
    """Add 'geometry', 'cshm', 'tau4' to the site dict (geometry None if not wanted)."""
    vecs = np.array([a["xyz"] - site["metal"]["xyz"] for a in site["ligands"]])
    site["geometry"], site["cshm"], site["tau4"] = None, None, None
    if site["cn"] == 4:
        s_td, s_sp = cshm(vecs, IDEAL["Td"]), cshm(vecs, IDEAL["SP"])
        site["tau4"] = tau4(bond_angles(vecs))
        geom, s = ("Td", s_td) if s_td <= s_sp else ("SP", s_sp)
        site["cshm"] = s
        if s <= MAX_CSHM:
            site["geometry"] = geom
    elif site["cn"] == 6:
        s = cshm(vecs, IDEAL["Oh"])
        site["cshm"] = s
        if s <= MAX_CSHM:
            site["geometry"] = "Oh"
    return site


def sort_geometries(sites):
    """Step 3: classify CN4 / CN6 sites and copy them into geometry folders."""
    selected = []
    for site in sites:
        if site["cn"] not in (4, 6):
            continue
        classify_site(site)
        if site["geometry"] is None:
            continue
        folder = os.path.join(OUT_DIR, f"CN{site['cn']}_{site['geometry']}")
        os.makedirs(folder, exist_ok=True)
        shutil.copy(os.path.join(OUT_DIR, "lig_files", site["file"]), folder)
        selected.append(site)
    for g in ("CN4_Td", "CN4_SP", "CN6_Oh"):
        n = sum(1 for s in selected if f"CN{s['cn']}_{s['geometry']}" == g)
        print(f"Step 3: {g}: {n} sites")
    return selected


# ---------------------------------------------------------------- step 4
def ligand_statistics(selected):
    """Step 4: residue + atom of every ligand, per CN/geometry. Writes two CSVs."""
    groups = {}
    for s in selected:
        groups.setdefault(f"CN{s['cn']}_{s['geometry']}", []).append(s)

    # per-site table
    with open(os.path.join(OUT_DIR, "sites.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["file", "pdb", "CN", "geometry", "CShM", "tau4",
                    "ligand_set", "ligands (residue chain-number atom: distance)"])
        for s in selected:
            ligset = "".join(f"{r}{n}" for r, n in
                             sorted(Counter(a["resname"] for a in s["ligands"]).items()))
            detail = "; ".join(f"{a['resname']} {a['chain']}{a['resseq']} {a['name']}: {a['dist']:.2f}"
                               for a in s["ligands"])
            w.writerow([s["file"], s["pdb"], s["cn"], s["geometry"], f"{s['cshm']:.3f}",
                        "" if s["tau4"] is None else f"{s['tau4']:.3f}", ligset, detail])

    # counts of residue / atom per group
    with open(os.path.join(OUT_DIR, "ligand_statistics.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["group", "residue", "atom", "count", "percent_of_ligands"])
        for g in sorted(groups):
            counts = Counter((a["resname"], a["name"]) for s in groups[g] for a in s["ligands"])
            total = sum(counts.values())
            print(f"\nStep 4: {g}  ({len(groups[g])} sites, {total} ligand atoms)")
            for (res, atom), n in counts.most_common():
                w.writerow([g, res, atom, n, f"{100 * n / total:.1f}"])
                print(f"   {res:<4} {atom:<4} {n:6d}  {100 * n / total:5.1f}%")


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    pdb_paths = collect_structures()
    sites = extract_sites(pdb_paths)
    selected = sort_geometries(sites)
    ligand_statistics(selected)
    print(f"\nDone. Results in {os.path.abspath(OUT_DIR)}")


if __name__ == "__main__":
    main()

RCSB: 7248 entries with ZN at <= 1.8 Å
  downloading 1IUJ (200/200)
Step 1: 200 structures contain ZN with resolution <= 1.8 Å
Step 2: 522 sites written to metal_sites_out/lig_files  (97 skipped)
Step 3: CN4_Td: 344 sites
Step 3: CN4_SP: 0 sites
Step 3: CN6_Oh: 6 sites

Step 4: CN4_Td  (344 sites, 1376 ligand atoms)
   CYS  SG     1001   72.7%
   HIS  ND1     245   17.8%
   HIS  NE2      61    4.4%
   ASP  OD1      33    2.4%
   HOH  O        19    1.4%
   ASP  OD2       9    0.7%
   GLU  OE2       3    0.2%
   GLU  OE1       2    0.1%
   SER  OG        1    0.1%
   LEU  OXT       1    0.1%
   ARG  NH1       1    0.1%

Step 4: CN6_Oh  (6 sites, 36 ligand atoms)
   HOH  O        21   58.3%
   HIS  NE2       6   16.7%
   ASP  OD2       4   11.1%
   THR  OG1       2    5.6%
   ASP  OD1       1    2.8%
   GLU  OE2       1    2.8%
   GLU  OE1       1    2.8%

Done. Results in /content/metal_sites_out


In [7]:


# =============================== SETTINGS ===================================
METAL = "CO"               # element symbol of the metal, e.g. "MN", "FE", "CU", "ZN"
MAX_RESOLUTION = 1.8       # Å
DOWNLOAD = True            # True: search + download from RCSB; False: use PDB_DIR only
MAX_DOWNLOADS = 200        # limit on how many structures to download (None = all)
PDB_DIR = "pdb_files"      # where .pdb files are (or will be downloaded to)
OUT_DIR = "metal_sites_out"
CUTOFF = 2.8               # Å, max metal–donor distance counted as coordinating
DONORS = {"N", "O", "S", "SE"}
ONLY_PROTEIN_AND_WATER = True  # skip sites with any non-amino-acid / non-water ligand
MAX_CSHM = 3.0             # sites worse than this are not assigned Td / SP / Oh

# RCSB chemical component codes for each metal's ions (used for the search)
ION_CODES = {
    "MN": ["MN", "MN3"], "FE": ["FE", "FE2"], "CO": ["CO", "3CO"], "NI": ["NI", "3NI"],
    "CU": ["CU", "CU1"], "ZN": ["ZN"], "V": ["V"], "CR": ["CR"], "TI": ["TI"],
}
WATER = {"HOH", "WAT", "DOD"}
AMINO_ACIDS = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
    "MSE", "SEC", "CSO", "CSD", "KCX", "SEP", "TPO", "PTR", "OCS", "MLY",
}
# ============================================================================


# ---------------------------------------------------------------- step 1
def search_rcsb(metal, max_res):
    """Return PDB IDs (X-ray) containing the metal ion with resolution <= max_res."""
    codes = ION_CODES.get(metal.upper(), [metal.upper()])
    query = {
        "query": {"type": "group", "logical_operator": "and", "nodes": [
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_entry_info.resolution_combined",
                "operator": "less_or_equal", "value": max_res}},
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_nonpolymer_entity_container_identifiers.nonpolymer_comp_id",
                "operator": "in", "value": codes}},
        ]},
        "return_type": "entry",
        "request_options": {"return_all_hits": True},
    }
    req = urllib.request.Request("https://search.rcsb.org/rcsbsearch/v2/query",
                                 data=json.dumps(query).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            if resp.status == 204:          # query OK but nothing found
                return []
            data = json.load(resp)
    except urllib.error.HTTPError as e:
        print("RCSB search failed:", e.code, e.read().decode(errors="replace")[:500])
        return []
    return [hit["identifier"] for hit in data.get("result_set", [])]


def download_pdb(pdb_id, folder):
    """Download one structure: .pdb if it exists, otherwise .cif (mmCIF).
    Many newer / large entries are only distributed as mmCIF."""
    for ext in ("pdb", "cif"):
        path = os.path.join(folder, f"{pdb_id.lower()}.{ext}")
        if os.path.exists(path):
            return path
    for ext in ("pdb", "cif"):
        path = os.path.join(folder, f"{pdb_id.lower()}.{ext}")
        url = f"https://files.rcsb.org/download/{pdb_id.upper()}.{ext}"
        try:
            urllib.request.urlretrieve(url, path)
            return path
        except urllib.error.HTTPError as e:
            if e.code != 404:
                print(f"  could not download {pdb_id}.{ext}: {e}")
        except Exception as e:
            print(f"  could not download {pdb_id}.{ext}: {e}")
    print(f"  {pdb_id}: not available as .pdb or .cif")
    return None


# ---- mmCIF reading (simple: only what we need) ----
def cif_tokens(line):
    """Split an mmCIF data line into tokens, respecting 'quoted' and "quoted" values."""
    return [t[1:-1] if t[:1] in "'\"" else t
            for t in re.findall(r"'[^']*'(?=\s|$)|\"[^\"]*\"(?=\s|$)|\S+", line)]


def cif_value(path, keys):
    """Return the first numeric value found for any of the given single-value keys."""
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if len(parts) >= 2 and parts[0] in keys:
                try:
                    return float(parts[1])
                except ValueError:
                    pass
    return None


def pdb_line(record, serial, name, alt, resname, chain, resseq, icode, xyz, occ, b, elem):
    """Write one atom as a PDB-format line (used for atoms read from mmCIF)."""
    name = name if len(name) == 4 or len(elem) == 2 else " " + name
    return (f"{record:<6}{serial % 100000:5d} {name:<4}{alt or ' ':1}{resname:>3} "
            f"{chain[:1]:1}{resseq:>4}{icode or ' ':1}   {xyz[0]:8.3f}{xyz[1]:8.3f}{xyz[2]:8.3f}"
            f"{occ:6.2f}{b:6.2f}          {elem:>2}")


def read_atoms_cif(path):
    """Read _atom_site from an mmCIF file (first model, first alt conformer)."""
    atoms, alt_seen, cols, in_loop, model1 = [], {}, [], False, None
    with open(path) as fh:
        for line in fh:
            s = line.strip()
            if s.startswith("_atom_site."):
                cols.append(s.split()[0][len("_atom_site."):])
                in_loop = True
                continue
            if not in_loop or not cols:
                continue
            if not s or s.startswith(("#", "loop_", "_")):
                if atoms or s.startswith(("#", "loop_")):
                    break
                continue
            v = dict(zip(cols, cif_tokens(s)))
            if v.get("group_PDB") not in ("ATOM", "HETATM"):
                continue
            model = v.get("pdbx_PDB_model_num", "1")
            model1 = model1 or model
            if model != model1:
                break
            clean = lambda x: "" if x in (".", "?") else x
            name = v.get("auth_atom_id", v.get("label_atom_id"))
            resname = v.get("auth_comp_id", v.get("label_comp_id"))
            chain = v.get("auth_asym_id", v.get("label_asym_id"))
            resseq = v.get("auth_seq_id", v.get("label_seq_id"))
            icode = clean(v.get("pdbx_PDB_ins_code", ""))
            alt = clean(v.get("label_alt_id", ""))
            key = (chain, resseq + icode, resname, name)
            if alt and alt_seen.setdefault(key, alt) != alt:
                continue
            elem = v.get("type_symbol", name[:1]).upper()
            xyz = np.array([float(v["Cartn_x"]), float(v["Cartn_y"]), float(v["Cartn_z"])])
            occ = float(clean(v.get("occupancy", "1")) or 1)
            b = float(clean(v.get("B_iso_or_equiv", "0")) or 0)
            atoms.append({
                "line": pdb_line(v["group_PDB"], int(v.get("id", 0)), name, alt, resname,
                                 chain, resseq, icode, xyz, occ, b, elem),
                "record": v["group_PDB"], "name": name, "resname": resname,
                "chain": chain, "resseq": resseq + icode, "element": elem, "xyz": xyz,
            })
    return atoms


def get_resolution(pdb_path):
    """Resolution in Å; None if absent (e.g. NMR). Works for .pdb and .cif."""
    if pdb_path.lower().endswith(".cif"):
        return cif_value(pdb_path, ("_refine.ls_d_res_high", "_reflns.d_resolution_high",
                                    "_em_3d_reconstruction.resolution"))
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith("REMARK   2 RESOLUTION."):
                try:
                    return float(line[23:30])
                except ValueError:
                    return None
            if line.startswith("ATOM"):
                break
    return None


def read_atoms(pdb_path):
    """Return a list of atom dicts from the first model, first alt conformer only."""
    if pdb_path.lower().endswith(".cif"):
        return read_atoms_cif(pdb_path)
    atoms, alt_seen = [], {}
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith("ENDMDL"):
                break
            if not line.startswith(("ATOM", "HETATM")):
                continue
            alt = line[16].strip()
            name = line[12:16].strip()
            key = (line[21], line[22:27], line[17:20], name)
            if alt and alt_seen.setdefault(key, alt) != alt:
                continue
            elem = line[76:78].strip().upper()
            if not elem:
                elem = "".join(c for c in name if c.isalpha())[:2].upper()
            atoms.append({
                "line": line.rstrip("\n"), "record": line[:6].strip(), "name": name,
                "resname": line[17:20].strip(), "chain": line[21].strip(),
                "resseq": line[22:27].strip(), "element": elem,
                "xyz": np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])]),
            })
    return atoms


def contains_metal(atoms, metal):
    return any(a["record"] == "HETATM" and a["element"] == metal for a in atoms)


def collect_structures():
    """Step 1: return list of .pdb paths with the metal and resolution <= cutoff."""
    os.makedirs(PDB_DIR, exist_ok=True)
    if DOWNLOAD:
        ids = search_rcsb(METAL, MAX_RESOLUTION)
        print(f"RCSB: {len(ids)} entries with {METAL} at <= {MAX_RESOLUTION} Å")
        if MAX_DOWNLOADS:
            ids = ids[:MAX_DOWNLOADS]
        for i, pid in enumerate(ids, 1):
            print(f"  downloading {pid} ({i}/{len(ids)})", end="\r")
            download_pdb(pid, PDB_DIR)
        print()

    kept = []
    for fn in sorted(os.listdir(PDB_DIR)):
        if not fn.lower().endswith((".pdb", ".ent", ".cif")):
            continue
        path = os.path.join(PDB_DIR, fn)
        res = get_resolution(path)
        if res is None or res > MAX_RESOLUTION:
            continue
        if contains_metal(read_atoms(path), METAL):
            kept.append(path)
    print(f"Step 1: {len(kept)} structures contain {METAL} with resolution <= {MAX_RESOLUTION} Å")
    return kept


# ---------------------------------------------------------------- step 2
def find_sites(atoms, metal, cutoff):
    """Return list of (metal_atom, [ligand atoms sorted by distance])."""
    xyz = np.array([a["xyz"] for a in atoms])
    sites = []
    for i, m in enumerate(atoms):
        if m["record"] != "HETATM" or m["element"] != metal:
            continue
        d = np.linalg.norm(xyz - m["xyz"], axis=1)
        ligs = []
        for j in np.where(d <= cutoff)[0]:
            if j != i and atoms[j]["element"] in DONORS:
                ligs.append(dict(atoms[j], dist=float(d[j])))
        ligs.sort(key=lambda a: a["dist"])
        sites.append((m, ligs))
    return sites


def ligands_allowed(ligs):
    for a in ligs:
        if a["resname"] not in AMINO_ACIDS and a["resname"] not in WATER:
            return False
    return True


def write_lig(metal_atom, ligs, path):
    with open(path, "w") as fh:
        fh.write(metal_atom["line"] + "\n")
        for a in ligs:
            fh.write(a["line"] + "\n")


def unique_name(folder, base):
    """pdbid_M_CN.lig, or pdbid_M_CN_2.lig, _3 ... if a site with that name exists."""
    name, k = f"{base}.lig", 2
    while os.path.exists(os.path.join(folder, name)):
        name, k = f"{base}_{k}.lig", k + 1
    return name


def extract_sites(pdb_paths):
    """Step 2: write .lig files; return a list of site dicts."""
    lig_dir = os.path.join(OUT_DIR, "lig_files")
    os.makedirs(lig_dir, exist_ok=True)
    sites, skipped = [], 0
    for path in pdb_paths:
        pdb_id = os.path.basename(path).split(".")[0].lower()
        for metal_atom, ligs in find_sites(read_atoms(path), METAL, CUTOFF):
            if not ligs or (ONLY_PROTEIN_AND_WATER and not ligands_allowed(ligs)):
                skipped += 1
                continue
            cn = len(ligs)
            fname = unique_name(lig_dir, f"{pdb_id}_{METAL}_{cn}")
            write_lig(metal_atom, ligs, os.path.join(lig_dir, fname))
            sites.append({"file": fname, "pdb": pdb_id, "cn": cn,
                          "metal": metal_atom, "ligands": ligs})
    print(f"Step 2: {len(sites)} sites written to {lig_dir}  ({skipped} skipped)")
    return sites


# ---------------------------------------------------------------- step 3
def unit(v):
    v = np.asarray(v, float)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


IDEAL = {
    "Td": unit([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]),
    "SP": unit([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0]]),
    "Oh": unit([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]]),
}


def cshm(vectors, ideal):
    """Continuous shape measure (metal included). 0 = perfect shape."""
    Q = np.vstack([np.zeros(3), vectors])
    Q = Q - Q.mean(0)
    best = np.inf
    for perm in itertools.permutations(range(len(vectors))):
        P = np.vstack([np.zeros(3), ideal[list(perm)]])
        P = P - P.mean(0)
        U, S, Vt = np.linalg.svd(P.T @ Q)
        d = np.sign(np.linalg.det(Vt.T @ U.T))
        tr = S[0] + S[1] + d * S[2]
        best = min(best, 100 * (1 - tr ** 2 / ((Q ** 2).sum() * (P ** 2).sum())))
    return max(best, 0.0)


def bond_angles(vectors):
    out = []
    for u, v in itertools.combinations(vectors, 2):
        c = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
        out.append(float(np.degrees(np.arccos(np.clip(c, -1, 1)))))
    return out


def tau4(angles):
    a = sorted(angles, reverse=True)
    return (360 - (a[0] + a[1])) / 141          # 1 = Td, 0 = SP


def classify_site(site):
    """Add 'geometry', 'cshm', 'tau4' to the site dict (geometry None if not wanted)."""
    vecs = np.array([a["xyz"] - site["metal"]["xyz"] for a in site["ligands"]])
    site["geometry"], site["cshm"], site["tau4"] = None, None, None
    if site["cn"] == 4:
        s_td, s_sp = cshm(vecs, IDEAL["Td"]), cshm(vecs, IDEAL["SP"])
        site["tau4"] = tau4(bond_angles(vecs))
        geom, s = ("Td", s_td) if s_td <= s_sp else ("SP", s_sp)
        site["cshm"] = s
        if s <= MAX_CSHM:
            site["geometry"] = geom
    elif site["cn"] == 6:
        s = cshm(vecs, IDEAL["Oh"])
        site["cshm"] = s
        if s <= MAX_CSHM:
            site["geometry"] = "Oh"
    return site


def sort_geometries(sites):
    """Step 3: classify CN4 / CN6 sites and copy them into geometry folders."""
    selected = []
    for site in sites:
        if site["cn"] not in (4, 6):
            continue
        classify_site(site)
        if site["geometry"] is None:
            continue
        folder = os.path.join(OUT_DIR, f"CN{site['cn']}_{site['geometry']}")
        os.makedirs(folder, exist_ok=True)
        shutil.copy(os.path.join(OUT_DIR, "lig_files", site["file"]), folder)
        selected.append(site)
    for g in ("CN4_Td", "CN4_SP", "CN6_Oh"):
        n = sum(1 for s in selected if f"CN{s['cn']}_{s['geometry']}" == g)
        print(f"Step 3: {g}: {n} sites")
    return selected


# ---------------------------------------------------------------- step 4
def ligand_statistics(selected):
    """Step 4: residue + atom of every ligand, per CN/geometry. Writes two CSVs."""
    groups = {}
    for s in selected:
        groups.setdefault(f"CN{s['cn']}_{s['geometry']}", []).append(s)

    # per-site table
    with open(os.path.join(OUT_DIR, "sites.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["file", "pdb", "CN", "geometry", "CShM", "tau4",
                    "ligand_set", "ligands (residue chain-number atom: distance)"])
        for s in selected:
            ligset = "".join(f"{r}{n}" for r, n in
                             sorted(Counter(a["resname"] for a in s["ligands"]).items()))
            detail = "; ".join(f"{a['resname']} {a['chain']}{a['resseq']} {a['name']}: {a['dist']:.2f}"
                               for a in s["ligands"])
            w.writerow([s["file"], s["pdb"], s["cn"], s["geometry"], f"{s['cshm']:.3f}",
                        "" if s["tau4"] is None else f"{s['tau4']:.3f}", ligset, detail])

    # counts of residue / atom per group
    with open(os.path.join(OUT_DIR, "ligand_statistics.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["group", "residue", "atom", "count", "percent_of_ligands"])
        for g in sorted(groups):
            counts = Counter((a["resname"], a["name"]) for s in groups[g] for a in s["ligands"])
            total = sum(counts.values())
            print(f"\nStep 4: {g}  ({len(groups[g])} sites, {total} ligand atoms)")
            for (res, atom), n in counts.most_common():
                w.writerow([g, res, atom, n, f"{100 * n / total:.1f}"])
                print(f"   {res:<4} {atom:<4} {n:6d}  {100 * n / total:5.1f}%")


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    pdb_paths = collect_structures()
    sites = extract_sites(pdb_paths)
    selected = sort_geometries(sites)
    ligand_statistics(selected)
    print(f"\nDone. Results in {os.path.abspath(OUT_DIR)}")


if __name__ == "__main__":
    main()

RCSB: 472 entries with CO at <= 1.8 Å

Step 1: 200 structures contain CO with resolution <= 1.8 Å
Step 2: 345 sites written to metal_sites_out/lig_files  (194 skipped)
Step 3: CN4_Td: 8 sites
Step 3: CN4_SP: 0 sites
Step 3: CN6_Oh: 95 sites

Step 4: CN4_Td  (8 sites, 32 ligand atoms)
   HIS  ND1      10   31.2%
   CYS  SG        9   28.1%
   HIS  NE2       7   21.9%
   HOH  O         6   18.8%

Step 4: CN6_Oh  (95 sites, 570 ligand atoms)
   HOH  O       254   44.6%
   HIS  NE2      95   16.7%
   ASP  OD2      46    8.1%
   GLU  OE1      32    5.6%
   GLU  OE2      20    3.5%
   ASP  OD1      18    3.2%
   HIS  ND1      14    2.5%
   CSD  SG       13    2.3%
   ASP  O        12    2.1%
   SER  N         8    1.4%
   CYS  SG        8    1.4%
   TYR  OH        7    1.2%
   KCX  OQ2       6    1.1%
   CSD  N         5    0.9%
   THR  OG1       4    0.7%
   PHE  O         4    0.7%
   ASN  OD1       4    0.7%
   CSO  N         3    0.5%
   CSO  SG        3    0.5%
   KCX  OQ1       3    0.

In [9]:
#!/usr/bin/env python3
"""
Metal-site pipeline (functions only, no classes)

 1. Get PDB structures that contain metal M with resolution <= MAX_RESOLUTION
    (either download them from RCSB, or scan a folder of .pdb files you have)
 2. Cut every M coordination site out into a .lig file:  pdbid_M_CN.lig
 3. CN 4  -> sort into tetrahedral (Td) and square planar (SP)
    CN 6  -> keep octahedral (Oh)
 4. For each CN / geometry, count which residues and atoms are the ligands

Only numpy is needed.  Edit the SETTINGS block and run:
    python metal_sites.py
"""
import csv
import itertools
import json
import os
import re
import shutil
import urllib.error
import urllib.request
from collections import Counter

import numpy as np

# =============================== SETTINGS ===================================
METAL = "NI"               # element symbol of the metal, e.g. "MN", "FE", "CU", "ZN"
MAX_RESOLUTION = 1.8       # Å
DOWNLOAD = True            # True: search + download from RCSB; False: use PDB_DIR only
MAX_DOWNLOADS = 200        # limit on how many structures to download (None = all)
PDB_DIR = "pdb_files"      # where .pdb files are (or will be downloaded to)
OUT_DIR = "metal_sites_out"
CUTOFF = 2.8               # Å, max metal–donor distance counted as coordinating
DONORS = {"N", "O", "S", "SE"}
ONLY_PROTEIN_AND_WATER = True  # skip sites with any non-amino-acid / non-water ligand
MAX_CSHM = 3.0             # sites worse than this are not assigned Td / SP / Oh

# RCSB chemical component codes for each metal's ions (used for the search)
ION_CODES = {
    "MN": ["MN", "MN3"], "FE": ["FE", "FE2"], "CO": ["CO", "3CO"], "NI": ["NI", "3NI"],
    "CU": ["CU", "CU1"], "ZN": ["ZN"], "V": ["V"], "CR": ["CR"], "TI": ["TI"],
}
WATER = {"HOH", "WAT", "DOD"}
AMINO_ACIDS = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
    "MSE", "SEC", "CSO", "CSD", "KCX", "SEP", "TPO", "PTR", "OCS", "MLY",
}
# ============================================================================


# ---------------------------------------------------------------- step 1
def search_rcsb(metal, max_res):
    """Return PDB IDs (X-ray) containing the metal ion with resolution <= max_res."""
    codes = ION_CODES.get(metal.upper(), [metal.upper()])
    query = {
        "query": {"type": "group", "logical_operator": "and", "nodes": [
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_entry_info.resolution_combined",
                "operator": "less_or_equal", "value": max_res}},
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_nonpolymer_entity_container_identifiers.nonpolymer_comp_id",
                "operator": "in", "value": codes}},
        ]},
        "return_type": "entry",
        "request_options": {"return_all_hits": True},
    }
    req = urllib.request.Request("https://search.rcsb.org/rcsbsearch/v2/query",
                                 data=json.dumps(query).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            if resp.status == 204:          # query OK but nothing found
                return []
            data = json.load(resp)
    except urllib.error.HTTPError as e:
        print("RCSB search failed:", e.code, e.read().decode(errors="replace")[:500])
        return []
    return [hit["identifier"] for hit in data.get("result_set", [])]


def download_pdb(pdb_id, folder):
    """Download one structure: .pdb if it exists, otherwise .cif (mmCIF).
    Many newer / large entries are only distributed as mmCIF."""
    for ext in ("pdb", "cif"):
        path = os.path.join(folder, f"{pdb_id.lower()}.{ext}")
        if os.path.exists(path):
            return path
    for ext in ("pdb", "cif"):
        path = os.path.join(folder, f"{pdb_id.lower()}.{ext}")
        url = f"https://files.rcsb.org/download/{pdb_id.upper()}.{ext}"
        try:
            urllib.request.urlretrieve(url, path)
            return path
        except urllib.error.HTTPError as e:
            if e.code != 404:
                print(f"  could not download {pdb_id}.{ext}: {e}")
        except Exception as e:
            print(f"  could not download {pdb_id}.{ext}: {e}")
    print(f"  {pdb_id}: not available as .pdb or .cif")
    return None


# ---- mmCIF reading (simple: only what we need) ----
def cif_tokens(line):
    """Split an mmCIF data line into tokens, respecting 'quoted' and "quoted" values."""
    return [t[1:-1] if t[:1] in "'\"" else t
            for t in re.findall(r"'[^']*'(?=\s|$)|\"[^\"]*\"(?=\s|$)|\S+", line)]


def cif_value(path, keys):
    """Return the first numeric value found for any of the given single-value keys."""
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if len(parts) >= 2 and parts[0] in keys:
                try:
                    return float(parts[1])
                except ValueError:
                    pass
    return None


def pdb_line(record, serial, name, alt, resname, chain, resseq, icode, xyz, occ, b, elem):
    """Write one atom as a PDB-format line (used for atoms read from mmCIF)."""
    name = name if len(name) == 4 or len(elem) == 2 else " " + name
    return (f"{record:<6}{serial % 100000:5d} {name:<4}{alt or ' ':1}{resname:>3} "
            f"{chain[:1]:1}{resseq:>4}{icode or ' ':1}   {xyz[0]:8.3f}{xyz[1]:8.3f}{xyz[2]:8.3f}"
            f"{occ:6.2f}{b:6.2f}          {elem:>2}")


def read_atoms_cif(path):
    """Read _atom_site from an mmCIF file (first model, first alt conformer)."""
    atoms, alt_seen, cols, in_loop, model1 = [], {}, [], False, None
    with open(path) as fh:
        for line in fh:
            s = line.strip()
            if s.startswith("_atom_site."):
                cols.append(s.split()[0][len("_atom_site."):])
                in_loop = True
                continue
            if not in_loop or not cols:
                continue
            if not s or s.startswith(("#", "loop_", "_")):
                if atoms or s.startswith(("#", "loop_")):
                    break
                continue
            v = dict(zip(cols, cif_tokens(s)))
            if v.get("group_PDB") not in ("ATOM", "HETATM"):
                continue
            model = v.get("pdbx_PDB_model_num", "1")
            model1 = model1 or model
            if model != model1:
                break
            clean = lambda x: "" if x in (".", "?") else x
            name = v.get("auth_atom_id", v.get("label_atom_id"))
            resname = v.get("auth_comp_id", v.get("label_comp_id"))
            chain = v.get("auth_asym_id", v.get("label_asym_id"))
            resseq = v.get("auth_seq_id", v.get("label_seq_id"))
            icode = clean(v.get("pdbx_PDB_ins_code", ""))
            alt = clean(v.get("label_alt_id", ""))
            key = (chain, resseq + icode, resname, name)
            if alt and alt_seen.setdefault(key, alt) != alt:
                continue
            elem = v.get("type_symbol", name[:1]).upper()
            xyz = np.array([float(v["Cartn_x"]), float(v["Cartn_y"]), float(v["Cartn_z"])])
            occ = float(clean(v.get("occupancy", "1")) or 1)
            b = float(clean(v.get("B_iso_or_equiv", "0")) or 0)
            atoms.append({
                "line": pdb_line(v["group_PDB"], int(v.get("id", 0)), name, alt, resname,
                                 chain, resseq, icode, xyz, occ, b, elem),
                "record": v["group_PDB"], "name": name, "resname": resname,
                "chain": chain, "resseq": resseq + icode, "element": elem, "xyz": xyz,
            })
    return atoms


def get_resolution(pdb_path):
    """Resolution in Å; None if absent (e.g. NMR). Works for .pdb and .cif."""
    if pdb_path.lower().endswith(".cif"):
        return cif_value(pdb_path, ("_refine.ls_d_res_high", "_reflns.d_resolution_high",
                                    "_em_3d_reconstruction.resolution"))
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith("REMARK   2 RESOLUTION."):
                try:
                    return float(line[23:30])
                except ValueError:
                    return None
            if line.startswith("ATOM"):
                break
    return None


def read_atoms(pdb_path):
    """Return a list of atom dicts from the first model, first alt conformer only."""
    if pdb_path.lower().endswith(".cif"):
        return read_atoms_cif(pdb_path)
    atoms, alt_seen = [], {}
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith("ENDMDL"):
                break
            if not line.startswith(("ATOM", "HETATM")):
                continue
            alt = line[16].strip()
            name = line[12:16].strip()
            key = (line[21], line[22:27], line[17:20], name)
            if alt and alt_seen.setdefault(key, alt) != alt:
                continue
            elem = line[76:78].strip().upper()
            if not elem:
                elem = "".join(c for c in name if c.isalpha())[:2].upper()
            atoms.append({
                "line": line.rstrip("\n"), "record": line[:6].strip(), "name": name,
                "resname": line[17:20].strip(), "chain": line[21].strip(),
                "resseq": line[22:27].strip(), "element": elem,
                "xyz": np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])]),
            })
    return atoms


def contains_metal(atoms, metal):
    return any(a["record"] == "HETATM" and a["element"] == metal for a in atoms)


def collect_structures():
    """Step 1: return list of .pdb paths with the metal and resolution <= cutoff."""
    os.makedirs(PDB_DIR, exist_ok=True)
    if DOWNLOAD:
        ids = search_rcsb(METAL, MAX_RESOLUTION)
        print(f"RCSB: {len(ids)} entries with {METAL} at <= {MAX_RESOLUTION} Å")
        if MAX_DOWNLOADS:
            ids = ids[:MAX_DOWNLOADS]
        for i, pid in enumerate(ids, 1):
            print(f"  downloading {pid} ({i}/{len(ids)})", end="\r")
            download_pdb(pid, PDB_DIR)
        print()

    kept = []
    for fn in sorted(os.listdir(PDB_DIR)):
        if not fn.lower().endswith((".pdb", ".ent", ".cif")):
            continue
        path = os.path.join(PDB_DIR, fn)
        res = get_resolution(path)
        if res is None or res > MAX_RESOLUTION:
            continue
        if contains_metal(read_atoms(path), METAL):
            kept.append(path)
    print(f"Step 1: {len(kept)} structures contain {METAL} with resolution <= {MAX_RESOLUTION} Å")
    return kept


# ---------------------------------------------------------------- step 2
def find_sites(atoms, metal, cutoff):
    """Return list of (metal_atom, [ligand atoms sorted by distance])."""
    xyz = np.array([a["xyz"] for a in atoms])
    sites = []
    for i, m in enumerate(atoms):
        if m["record"] != "HETATM" or m["element"] != metal:
            continue
        d = np.linalg.norm(xyz - m["xyz"], axis=1)
        ligs = []
        for j in np.where(d <= cutoff)[0]:
            if j != i and atoms[j]["element"] in DONORS:
                ligs.append(dict(atoms[j], dist=float(d[j])))
        ligs.sort(key=lambda a: a["dist"])
        sites.append((m, ligs))
    return sites


def ligands_allowed(ligs):
    for a in ligs:
        if a["resname"] not in AMINO_ACIDS and a["resname"] not in WATER:
            return False
    return True


def write_lig(metal_atom, ligs, path):
    with open(path, "w") as fh:
        fh.write(metal_atom["line"] + "\n")
        for a in ligs:
            fh.write(a["line"] + "\n")


def unique_name(folder, base):
    """pdbid_M_CN.lig, or pdbid_M_CN_2.lig, _3 ... if a site with that name exists."""
    name, k = f"{base}.lig", 2
    while os.path.exists(os.path.join(folder, name)):
        name, k = f"{base}_{k}.lig", k + 1
    return name


def extract_sites(pdb_paths):
    """Step 2: write .lig files; return a list of site dicts."""
    lig_dir = os.path.join(OUT_DIR, "lig_files")
    os.makedirs(lig_dir, exist_ok=True)
    sites, skipped = [], 0
    for path in pdb_paths:
        pdb_id = os.path.basename(path).split(".")[0].lower()
        for metal_atom, ligs in find_sites(read_atoms(path), METAL, CUTOFF):
            if not ligs or (ONLY_PROTEIN_AND_WATER and not ligands_allowed(ligs)):
                skipped += 1
                continue
            cn = len(ligs)
            fname = unique_name(lig_dir, f"{pdb_id}_{METAL}_{cn}")
            write_lig(metal_atom, ligs, os.path.join(lig_dir, fname))
            sites.append({"file": fname, "pdb": pdb_id, "cn": cn,
                          "metal": metal_atom, "ligands": ligs})
    print(f"Step 2: {len(sites)} sites written to {lig_dir}  ({skipped} skipped)")
    return sites


# ---------------------------------------------------------------- step 3
def unit(v):
    v = np.asarray(v, float)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


IDEAL = {
    "Td": unit([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]),
    "SP": unit([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0]]),
    "Oh": unit([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]]),
}


def cshm(vectors, ideal):
    """Continuous shape measure (metal included). 0 = perfect shape."""
    Q = np.vstack([np.zeros(3), vectors])
    Q = Q - Q.mean(0)
    best = np.inf
    for perm in itertools.permutations(range(len(vectors))):
        P = np.vstack([np.zeros(3), ideal[list(perm)]])
        P = P - P.mean(0)
        U, S, Vt = np.linalg.svd(P.T @ Q)
        d = np.sign(np.linalg.det(Vt.T @ U.T))
        tr = S[0] + S[1] + d * S[2]
        best = min(best, 100 * (1 - tr ** 2 / ((Q ** 2).sum() * (P ** 2).sum())))
    return max(best, 0.0)


def bond_angles(vectors):
    out = []
    for u, v in itertools.combinations(vectors, 2):
        c = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
        out.append(float(np.degrees(np.arccos(np.clip(c, -1, 1)))))
    return out


def tau4(angles):
    a = sorted(angles, reverse=True)
    return (360 - (a[0] + a[1])) / 141          # 1 = Td, 0 = SP


def classify_site(site):
    """Add 'geometry', 'cshm', 'tau4' to the site dict (geometry None if not wanted)."""
    vecs = np.array([a["xyz"] - site["metal"]["xyz"] for a in site["ligands"]])
    site["geometry"], site["cshm"], site["tau4"] = None, None, None
    if site["cn"] == 4:
        s_td, s_sp = cshm(vecs, IDEAL["Td"]), cshm(vecs, IDEAL["SP"])
        site["tau4"] = tau4(bond_angles(vecs))
        geom, s = ("Td", s_td) if s_td <= s_sp else ("SP", s_sp)
        site["cshm"] = s
        if s <= MAX_CSHM:
            site["geometry"] = geom
    elif site["cn"] == 6:
        s = cshm(vecs, IDEAL["Oh"])
        site["cshm"] = s
        if s <= MAX_CSHM:
            site["geometry"] = "Oh"
    return site


def sort_geometries(sites):
    """Step 3: classify CN4 / CN6 sites and copy them into geometry folders."""
    selected = []
    for site in sites:
        if site["cn"] not in (4, 6):
            continue
        classify_site(site)
        if site["geometry"] is None:
            continue
        folder = os.path.join(OUT_DIR, f"CN{site['cn']}_{site['geometry']}")
        os.makedirs(folder, exist_ok=True)
        shutil.copy(os.path.join(OUT_DIR, "lig_files", site["file"]), folder)
        selected.append(site)
    for g in ("CN4_Td", "CN4_SP", "CN6_Oh"):
        n = sum(1 for s in selected if f"CN{s['cn']}_{s['geometry']}" == g)
        print(f"Step 3: {g}: {n} sites")
    return selected


# ---------------------------------------------------------------- step 4
def ligand_statistics(selected):
    """Step 4: residue + atom of every ligand, per CN/geometry. Writes two CSVs."""
    groups = {}
    for s in selected:
        groups.setdefault(f"CN{s['cn']}_{s['geometry']}", []).append(s)

    # per-site table
    with open(os.path.join(OUT_DIR, "sites.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["file", "pdb", "CN", "geometry", "CShM", "tau4",
                    "ligand_set", "ligands (residue chain-number atom: distance)"])
        for s in selected:
            ligset = "".join(f"{r}{n}" for r, n in
                             sorted(Counter(a["resname"] for a in s["ligands"]).items()))
            detail = "; ".join(f"{a['resname']} {a['chain']}{a['resseq']} {a['name']}: {a['dist']:.2f}"
                               for a in s["ligands"])
            w.writerow([s["file"], s["pdb"], s["cn"], s["geometry"], f"{s['cshm']:.3f}",
                        "" if s["tau4"] is None else f"{s['tau4']:.3f}", ligset, detail])

    # counts of residue / atom per group
    with open(os.path.join(OUT_DIR, "ligand_statistics.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["group", "residue", "atom", "count", "percent_of_ligands"])
        for g in sorted(groups):
            counts = Counter((a["resname"], a["name"]) for s in groups[g] for a in s["ligands"])
            total = sum(counts.values())
            print(f"\nStep 4: {g}  ({len(groups[g])} sites, {total} ligand atoms)")
            for (res, atom), n in counts.most_common():
                w.writerow([g, res, atom, n, f"{100 * n / total:.1f}"])
                print(f"   {res:<4} {atom:<4} {n:6d}  {100 * n / total:5.1f}%")


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    pdb_paths = collect_structures()
    sites = extract_sites(pdb_paths)
    selected = sort_geometries(sites)
    ligand_statistics(selected)
    print(f"\nDone. Results in {os.path.abspath(OUT_DIR)}")


if __name__ == "__main__":
    main()

RCSB: 1104 entries with NI at <= 1.8 Å
  downloading 4JZ4 (200/200)
Step 1: 202 structures contain NI with resolution <= 1.8 Å
Step 2: 296 sites written to metal_sites_out/lig_files  (108 skipped)
Step 3: CN4_Td: 4 sites
Step 3: CN4_SP: 46 sites
Step 3: CN6_Oh: 55 sites

Step 4: CN4_SP  (46 sites, 184 ligand atoms)
   CYS  SG       64   34.8%
   HIS  N        40   21.7%
   CYS  N        30   16.3%
   HIS  ND1      14    7.6%
   HIS  NE2      10    5.4%
   SER  N        10    5.4%
   GLY  N         9    4.9%
   HOH  O         6    3.3%
   GLN  N         1    0.5%

Step 4: CN4_Td  (4 sites, 16 ligand atoms)
   HIS  NE2      12   75.0%
   HOH  O         2   12.5%
   CYS  SG        2   12.5%

Step 4: CN6_Oh  (55 sites, 330 ligand atoms)
   HOH  O       158   47.9%
   HIS  NE2      98   29.7%
   ASP  OD1      17    5.2%
   GLU  OE1      11    3.3%
   HIS  ND1      11    3.3%
   GLU  OE2      10    3.0%
   ASP  OD2       9    2.7%
   HIS  N         3    0.9%
   MET  SD        2    0.6%
   GL